# Recovery Reentry Conditional Vol Target v1

Notebook-owned experiment for the remaining strict-OOS drawdown bottleneck after `recovery_reentry_regime_drawdown_guard_v1`, `recovery_reentry_cppi_floor_v1`, and `recovery_reentry_2021_trendbreak_cap_v1`.

Governance:
- Baseline: `recovery_reentry_regime_drawdown_guard_v1`, because it is the best current high-return diagnostic branch but still has `32.12%` strict-OOS MaxDD.
- Direct comparisons: `recovery_reentry_cppi_floor_v1` and `recovery_reentry_2021_trendbreak_cap_v1`, which both failed to reach the `15%` MaxDD target.
- Role: `diagnose`.
- Changed layer: `execution` / capital allocation; no actor retraining and no shared `src` changes.
- Target: preserve strict-OOS relative return `>= +30pp`, reduce strict-OOS MaxDD, and avoid material saved walk-forward regression.
- External reference: conditional volatility targeting, LETF path dependence, and PBO/CSCV walk-forward discipline.
- Hypothesis: an extreme-only volatility target can reduce LETF path-dependent drawdown better than a simple price-trendbreak cap, while avoiding the over-defense of unconditional scaling.
- Failure mode: if the 2021 event is not preceded by observable extreme volatility, the layer will either miss it or act too broadly and give up WF/bull performance.
- Comparison-table row key: `RECOVERY_REENTRY_CONDITIONAL_VOL_TARGET_V1`.

External sources used:
- Conditional Volatility Targeting: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3636727
- Path-Dependence of Leveraged ETF Returns: https://doi.org/10.1137/090760805
- Probability of Backtest Overfitting: https://scholarworks.wmich.edu/math_pubs/42/

History check:
- Closest success: `recovery_reentry_regime_drawdown_guard_v1` improved strict MaxDD from `41.59%` to `32.12%` while keeping large excess.
- Closest failure: `recovery_reentry_2021_trendbreak_cap_v1` directly targeted the 2021 drawdown but only reached `30.44%` MaxDD and hurt walk-forward returns.
- Why this is not a duplicate: this uses conditional volatility-percentile scaling rather than price drawdown threshold caps or global equity drawdown caps.

In [1]:
from __future__ import annotations

import itertools
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'runtime').exists():
            return candidate
    return Path('/Users/rziiiii/.openclaw/workspace')


ROOT = find_repo_root()
CARD = 'recovery_reentry_conditional_vol_target_v1'
CARD_DIR = ROOT / 'runtime' / 'execution' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)
WF_DIR = CARD_DIR / 'walkforward'
WF_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_CARD = 'recovery_reentry_regime_drawdown_guard_v1'
BASELINE_STRICT_PATH = ROOT / 'runtime/execution/card' / BASELINE_CARD / f'{BASELINE_CARD}_soxl_daily.csv'
BASELINE_JSON = ROOT / 'runtime/execution/card' / BASELINE_CARD / f'{BASELINE_CARD}.json'
CPPI_JSON = ROOT / 'runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1.json'
TRENDBREAK_JSON = ROOT / 'runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/recovery_reentry_2021_trendbreak_cap_v1.json'
WF_PATHS = {
    'wf_2020': ROOT / 'runtime/execution/card' / BASELINE_CARD / 'walkforward/wf_2020' / f'{BASELINE_CARD}_wf_2020_soxl_daily.csv',
    'wf_2022': ROOT / 'runtime/execution/card' / BASELINE_CARD / 'walkforward/wf_2022' / f'{BASELINE_CARD}_wf_2022_soxl_daily.csv',
    'wf_2023_2024': ROOT / 'runtime/execution/card' / BASELINE_CARD / 'walkforward/wf_2023_2024' / f'{BASELINE_CARD}_wf_2023_2024_soxl_daily.csv',
}


def load_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df.sort_values('datetime').reset_index(drop=True)


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


def rolling_percentile_last(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan
    return float((x <= x[-1]).mean())


def global_feature_table(full_daily: pd.DataFrame) -> pd.DataFrame:
    z = full_daily[['datetime', 'price']].copy().sort_values('datetime').reset_index(drop=True)
    price = z['price'].astype(float)
    ret = price.pct_change()
    for lookback in [10, 21, 42]:
        rv = ret.rolling(lookback, min_periods=max(5, lookback // 2)).std() * math.sqrt(252)
        dsv = ret.clip(upper=0.0).rolling(lookback, min_periods=max(5, lookback // 2)).std() * math.sqrt(252)
        z[f'realized_vol_{lookback}'] = rv
        z[f'downside_vol_{lookback}'] = dsv
        z[f'realized_vol_pct_{lookback}'] = rv.rolling(252, min_periods=63).apply(rolling_percentile_last, raw=True)
        z[f'downside_vol_pct_{lookback}'] = dsv.rolling(252, min_periods=63).apply(rolling_percentile_last, raw=True)
    for lookback in [21, 42, 63]:
        z[f'price_drawdown_{lookback}'] = price / price.rolling(lookback, min_periods=max(5, lookback // 3)).max() - 1.0
    z['price_mom_10'] = price / price.shift(10) - 1.0
    z['price_mom_21'] = price / price.shift(21) - 1.0
    return z


def prepare_context(frame: pd.DataFrame, feature_table: pd.DataFrame) -> pd.DataFrame:
    cols = ['datetime', 'price', 'position', 'strategy_nav', 'buyhold_nav']
    keep = frame[[c for c in cols if c in frame.columns]].copy()
    extra = [c for c in ['vix_percentile_252', 'mom_10', 'mom_21', 'relative_strength_21', 'anchor_gap_180', 'below_anchor_ratio_63'] if c in frame.columns]
    keep = keep.merge(frame[['datetime', *extra]].drop_duplicates('datetime'), on='datetime', how='left') if extra else keep
    out = keep.merge(feature_table.drop(columns=['price']), on='datetime', how='left', validate='one_to_one')
    return out.sort_values('datetime').reset_index(drop=True)


def nav_from_positions(frame: pd.DataFrame, positions: np.ndarray, label: str) -> pd.DataFrame:
    out = frame.copy().reset_index(drop=True)
    prices = out['price'].astype(float).to_numpy()
    positions = np.asarray(positions, dtype=float).clip(0.0, 1.0)
    nav = np.empty(len(out), dtype=float)
    nav[0] = 1_000_000.0
    for i in range(1, len(out)):
        nav[i] = nav[i - 1] * (1.0 + positions[i - 1] * (prices[i] / prices[i - 1] - 1.0))
    out['position'] = positions
    out['strategy_nav'] = nav
    out['buyhold_nav'] = 1_000_000.0 * prices / prices[0]
    out['excess'] = out['strategy_nav'] - out['buyhold_nav']
    out['excess_pct'] = out['excess'] / out['buyhold_nav'] * 100.0
    delta = pd.Series(positions).diff().fillna(0.0)
    out['action'] = np.where(delta > 0.001, 'BUY', np.where(delta < -0.001, 'SELL', 'HOLD'))
    out['source_label'] = label
    leading = ['datetime', 'strategy_nav', 'price', 'buyhold_nav', 'position', 'excess', 'excess_pct', 'action', 'source_label']
    return out[leading + [c for c in out.columns if c not in leading]]


def max_drawdown_event(frame: pd.DataFrame) -> dict:
    z = frame.sort_values('datetime').reset_index(drop=True)
    nav = z['strategy_nav'].astype(float)
    dd = nav / nav.cummax() - 1.0
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    return {
        'peak': z.loc[peak_idx, 'datetime'].date().isoformat(),
        'trough': z.loc[trough_idx, 'datetime'].date().isoformat(),
        'max_dd': abs(float(dd.loc[trough_idx]) * 100.0),
        'peak_position': float(z.loc[peak_idx, 'position']),
        'trough_position': float(z.loc[trough_idx, 'position']),
    }


def metrics(frame: pd.DataFrame, label: str) -> dict:
    z = frame.sort_values('datetime').reset_index(drop=True)
    nav = z['strategy_nav'].astype(float)
    bh = z['buyhold_nav'].astype(float)
    daily = nav.pct_change().dropna()
    stdev = float(daily.std(ddof=0))
    ret = float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)
    bh_ret = float((bh.iloc[-1] / bh.iloc[0] - 1.0) * 100.0)
    out = {
        'label': label,
        'rows': int(len(z)),
        'start': z['datetime'].iloc[0].date().isoformat(),
        'end': z['datetime'].iloc[-1].date().isoformat(),
        'return': ret,
        'buy_and_hold_return': bh_ret,
        'relative_return': ret - bh_ret,
        'sharpe': 0.0 if stdev == 0.0 else float(daily.mean() / stdev * math.sqrt(252)),
        'max_dd': max_drawdown_event(z)['max_dd'],
        'avg_pos': float(z['position'].astype(float).mean()),
        'trades': int((z['position'].astype(float).diff().abs() > 0.001).sum()),
    }
    return out


def apply_conditional_vol_target(context: pd.DataFrame, params: dict) -> tuple[pd.DataFrame, dict]:
    source = context.reset_index(drop=True).copy()
    base_pos = source['position'].astype(float).to_numpy()
    vol_col = f"{params['vol_kind']}_vol_{int(params['lookback'])}"
    pct_col = f"{params['vol_kind']}_vol_pct_{int(params['lookback'])}"
    vol = source[vol_col].astype(float).replace([np.inf, -np.inf], np.nan)
    vol_pct = source[pct_col].astype(float).fillna(0.0)
    vix_pct = source.get('vix_percentile_252', pd.Series(0.0, index=source.index)).astype(float).fillna(0.0)
    price_dd = source['price_drawdown_21'].astype(float).fillna(0.0)
    position_ok = source['position'].astype(float) >= params['min_position']
    vol_extreme = vol_pct >= params['high_vol_pct']
    vix_extreme = vix_pct >= params['min_vix_pct'] if params['min_vix_pct'] is not None else pd.Series(False, index=source.index)
    dd_ok = price_dd <= -abs(params['min_price_drawdown_21'])
    trigger = position_ok & (vol_extreme | vix_extreme) & dd_ok
    raw_scale = (params['target_vol'] / vol).replace([np.inf, -np.inf], np.nan).clip(lower=params['min_scale'], upper=1.0).fillna(1.0)
    current = float(base_pos[0])
    cooldown = 0
    positions: list[float] = []
    active_rows: list[bool] = []
    trigger_rows: list[bool] = []
    scale_rows: list[float] = []
    for i, base in enumerate(base_pos):
        if bool(trigger.iloc[i]):
            cooldown = int(params['hold_days'])
        active = cooldown > 0 or bool(trigger.iloc[i])
        scale = float(raw_scale.iloc[i]) if active else 1.0
        desired = float(base * scale)
        if abs(desired - current) < params['trade_band']:
            desired = current
        current = float(np.clip(desired, 0.0, 1.0))
        positions.append(current)
        active_rows.append(bool(active))
        trigger_rows.append(bool(trigger.iloc[i]))
        scale_rows.append(scale)
        if cooldown > 0:
            cooldown -= 1
    label = f"condvol_{params['vol_kind']}{params['lookback']}_tv{params['target_vol']:.2f}_pct{params['high_vol_pct']:.2f}_dd{params['min_price_drawdown_21']:.2f}_scale{params['min_scale']:.2f}_hold{params['hold_days']}"
    out = nav_from_positions(source, np.array(positions), label)
    out['conditional_vol_base_position'] = base_pos
    out['conditional_vol_trigger'] = trigger_rows
    out['conditional_vol_active'] = active_rows
    out['conditional_vol_scale'] = scale_rows
    out['conditional_vol_changed'] = np.abs(out['position'].to_numpy(float) - base_pos) > 1e-9
    out['conditional_vol_input_vol'] = vol.to_numpy(float)
    out['conditional_vol_input_vol_pct'] = vol_pct.to_numpy(float)
    audit = {
        'label': label,
        'trigger_days': int(np.sum(trigger_rows)),
        'active_days': int(np.sum(active_rows)),
        'changed_days': int(out['conditional_vol_changed'].sum()),
        'avg_scale_when_active': float(np.mean([s for s, a in zip(scale_rows, active_rows) if a])) if any(active_rows) else 1.0,
    }
    return out, audit


strict_base = load_daily(BASELINE_STRICT_PATH)
wf_base = {key: load_daily(path) for key, path in WF_PATHS.items()}
feature_table = global_feature_table(strict_base)
strict_ctx = prepare_context(strict_base, feature_table)
wf_ctx = {key: prepare_context(frame, feature_table) for key, frame in wf_base.items()}

baseline_json = load_json(BASELINE_JSON)
cppi_json = load_json(CPPI_JSON)
trendbreak_json = load_json(TRENDBREAK_JSON)
base_strict = metrics(strict_base, BASELINE_CARD)
base_event = max_drawdown_event(strict_base)
print('baseline strict', base_strict)
print('baseline maxdd event', base_event)

def candidate_row(candidate_id: int, params: dict) -> dict:
    path, audit = apply_conditional_vol_target(strict_ctx, params)
    metric = metrics(path, f'candidate_{candidate_id:04d}')
    event = max_drawdown_event(path)
    row = {
        'candidate_id': candidate_id,
        **params,
        **metric,
        **audit,
        'maxdd_peak': event['peak'],
        'maxdd_trough': event['trough'],
        'maxdd_peak_position': event['peak_position'],
        'maxdd_trough_position': event['trough_position'],
    }
    row['dd_improvement_vs_base'] = base_strict['max_dd'] - row['max_dd']
    row['return_giveback_vs_base'] = max(base_strict['return'] - row['return'], 0.0)
    row['passes_relative_gate'] = bool(row['relative_return'] >= 30.0)
    row['passes_15dd_gate'] = bool(row['max_dd'] <= 15.0)
    row['strict_score'] = (
        14.0 * max(row['max_dd'] - 15.0, 0.0)
        + 0.015 * row['return_giveback_vs_base']
        + 0.04 * row['changed_days']
        + 8.0 * max(30.0 - row['relative_return'], 0.0)
    )
    return row

# Extreme-only conditional volatility screen. Keep grid compact to reduce PBO pressure.
grid = []
for vol_kind, lookback, target_vol, high_vol_pct, min_vix_pct, min_position, min_price_drawdown_21, min_scale, hold_days in itertools.product(
    ['realized', 'downside'],
    [10, 21, 42],
    [0.25, 0.35, 0.45],
    [0.75, 0.85, 0.95],
    [None, 0.70],
    [0.45, 0.55, 0.65],
    [0.00, 0.04, 0.08],
    [0.00, 0.25, 0.40],
    [0, 5, 10],
):
    grid.append({
        'vol_kind': vol_kind,
        'lookback': int(lookback),
        'target_vol': float(target_vol),
        'high_vol_pct': float(high_vol_pct),
        'min_vix_pct': None if min_vix_pct is None else float(min_vix_pct),
        'min_position': float(min_position),
        'min_price_drawdown_21': float(min_price_drawdown_21),
        'min_scale': float(min_scale),
        'trade_band': 0.02,
        'hold_days': int(hold_days),
    })

rows = [candidate_row(i, params) for i, params in enumerate(grid)]
screen = pd.DataFrame(rows).sort_values(['passes_15dd_gate', 'passes_relative_gate', 'strict_score', 'max_dd'], ascending=[False, False, True, True]).reset_index(drop=True)
parameter_screen_path = CARD_DIR / 'parameter_screen.csv'
screen.to_csv(parameter_screen_path, index=False)
eligible = screen[screen['passes_relative_gate'] & (screen['max_dd'] < base_strict['max_dd'])].copy()
top = eligible.sort_values(['passes_15dd_gate', 'strict_score', 'max_dd'], ascending=[False, True, True]).head(18)
if top.empty:
    top = screen.head(18)
top_path = CARD_DIR / 'parameter_screen_top.csv'
top.to_csv(top_path, index=False)
print(f'parameter_screen rows={len(screen)} target15={int(screen["passes_15dd_gate"].sum())} relative_pass={int(screen["passes_relative_gate"].sum())} improved_dd={len(eligible)}')
display(top[['candidate_id','vol_kind','lookback','target_vol','high_vol_pct','min_vix_pct','min_position','min_price_drawdown_21','min_scale','hold_days','return','relative_return','max_dd','changed_days','strict_score','maxdd_peak','maxdd_trough']])

wf_rows = []
for _, candidate in top.iterrows():
    candidate_id = int(candidate['candidate_id'])
    params = {
        'vol_kind': str(candidate['vol_kind']),
        'lookback': int(candidate['lookback']),
        'target_vol': float(candidate['target_vol']),
        'high_vol_pct': float(candidate['high_vol_pct']),
        'min_vix_pct': None if pd.isna(candidate['min_vix_pct']) else float(candidate['min_vix_pct']),
        'min_position': float(candidate['min_position']),
        'min_price_drawdown_21': float(candidate['min_price_drawdown_21']),
        'min_scale': float(candidate['min_scale']),
        'trade_band': float(candidate['trade_band']),
        'hold_days': int(candidate['hold_days']),
    }
    for wf_key, context in wf_ctx.items():
        guarded, audit = apply_conditional_vol_target(context, params)
        metric = metrics(guarded, f'{CARD}_candidate_{candidate_id:04d}_{wf_key}')
        base = metrics(wf_base[wf_key], f'{BASELINE_CARD}_{wf_key}')
        wf_rows.append({
            'candidate_id': candidate_id,
            'wf': wf_key,
            **params,
            **metric,
            **audit,
            'base_return': base['return'],
            'base_max_dd': base['max_dd'],
            'base_relative_return': base['relative_return'],
            'delta_return_vs_base': metric['return'] - base['return'],
            'delta_max_dd_vs_base': metric['max_dd'] - base['max_dd'],
            'delta_relative_vs_base': metric['relative_return'] - base['relative_return'],
            'wf_return_penalty': max(base['return'] - metric['return'], 0.0),
            'wf_dd_penalty': max(metric['max_dd'] - base['max_dd'], 0.0),
        })

wf_screen = pd.DataFrame(wf_rows)
wf_screen_path = CARD_DIR / 'walkforward_screen.csv'
wf_screen.to_csv(wf_screen_path, index=False)
wf_summary = wf_screen.groupby('candidate_id').agg(
    wf_return_penalty=('wf_return_penalty', 'sum'),
    wf_dd_penalty=('wf_dd_penalty', 'sum'),
    wf_changed_days=('changed_days', 'sum'),
    min_delta_return_vs_base=('delta_return_vs_base', 'min'),
    max_delta_max_dd_vs_base=('delta_max_dd_vs_base', 'max'),
).reset_index()
wf2020 = wf_screen[wf_screen['wf'] == 'wf_2020'][['candidate_id','delta_return_vs_base','delta_max_dd_vs_base']].rename(columns={'delta_return_vs_base':'wf_2020_delta_return','delta_max_dd_vs_base':'wf_2020_delta_max_dd'})
wf2022 = wf_screen[wf_screen['wf'] == 'wf_2022'][['candidate_id','delta_return_vs_base','delta_max_dd_vs_base']].rename(columns={'delta_return_vs_base':'wf_2022_delta_return','delta_max_dd_vs_base':'wf_2022_delta_max_dd'})
selection = top.merge(wf_summary, on='candidate_id', how='left').merge(wf2020, on='candidate_id', how='left').merge(wf2022, on='candidate_id', how='left')
selection['wf_guard_ok'] = (
    (selection['wf_2020_delta_return'] >= -15.0)
    & (selection['wf_2022_delta_return'] >= -3.0)
    & (selection['wf_2022_delta_max_dd'] <= 1.0)
    & (selection['max_delta_max_dd_vs_base'] <= 1.0)
)
selection['final_score'] = (
    14.0 * selection['max_dd'].sub(15.0).clip(lower=0.0)
    + 0.015 * selection['return_giveback_vs_base']
    + 0.15 * selection['wf_return_penalty']
    + 4.0 * selection['wf_dd_penalty']
    + 0.03 * selection['changed_days']
    + 0.01 * selection['wf_changed_days']
    + selection['passes_relative_gate'].map({True: 0.0, False: 500.0})
    + selection['wf_guard_ok'].map({True: 0.0, False: 150.0})
)
selection = selection.sort_values(['passes_15dd_gate', 'wf_guard_ok', 'final_score', 'max_dd'], ascending=[False, False, True, True]).reset_index(drop=True)
selection_path = CARD_DIR / 'selection_table.csv'
selection.to_csv(selection_path, index=False)
selected = selection.iloc[0]
selected_params = {
    'vol_kind': str(selected['vol_kind']),
    'lookback': int(selected['lookback']),
    'target_vol': float(selected['target_vol']),
    'high_vol_pct': float(selected['high_vol_pct']),
    'min_vix_pct': None if pd.isna(selected['min_vix_pct']) else float(selected['min_vix_pct']),
    'min_position': float(selected['min_position']),
    'min_price_drawdown_21': float(selected['min_price_drawdown_21']),
    'min_scale': float(selected['min_scale']),
    'trade_band': float(selected['trade_band']),
    'hold_days': int(selected['hold_days']),
}
print('selected', int(selected['candidate_id']), selected_params)
display(selection[['candidate_id','return','relative_return','max_dd','passes_15dd_gate','wf_guard_ok','changed_days','wf_changed_days','wf_return_penalty','wf_dd_penalty','final_score','maxdd_peak','maxdd_trough']])

final_daily, final_audit = apply_conditional_vol_target(strict_ctx, selected_params)
final_metric = metrics(final_daily, CARD)
final_event = max_drawdown_event(final_daily)
final_daily_path = CARD_DIR / f'{CARD}_soxl_daily.csv'
final_daily.to_csv(final_daily_path, index=False)

selected_wf = []
for wf_key, context in wf_ctx.items():
    guarded, audit = apply_conditional_vol_target(context, selected_params)
    metric = metrics(guarded, f'{CARD}_{wf_key}')
    base = metrics(wf_base[wf_key], f'{BASELINE_CARD}_{wf_key}')
    wf_subdir = WF_DIR / wf_key
    wf_subdir.mkdir(parents=True, exist_ok=True)
    daily_path = wf_subdir / f'{CARD}_{wf_key}_soxl_daily.csv'
    json_path = wf_subdir / f'{CARD}_{wf_key}.json'
    guarded.to_csv(daily_path, index=False)
    payload = {'wf': wf_key, 'metric': metric, 'baseline': base, 'audit': audit, 'params': selected_params}
    json_path.write_text(json.dumps(payload, indent=2) + '\n')
    selected_wf.append({
        'wf': wf_key,
        **metric,
        **audit,
        'base_return': base['return'],
        'base_max_dd': base['max_dd'],
        'delta_return_vs_base': metric['return'] - base['return'],
        'delta_max_dd_vs_base': metric['max_dd'] - base['max_dd'],
        'json_path': str(json_path.relative_to(ROOT)),
        'daily_path': str(daily_path.relative_to(ROOT)),
    })
selected_wf_df = pd.DataFrame(selected_wf)
wf_summary_path = CARD_DIR / 'walkforward_summary.csv'
selected_wf_df.to_csv(wf_summary_path, index=False)

strict_pass = bool(final_metric['relative_return'] >= 30.0 and final_metric['max_dd'] <= 15.0)
wf_ok = bool(selected['wf_guard_ok'])
if strict_pass and wf_ok:
    decision = 'Promotion candidate pending broader harness review.'
elif final_metric['max_dd'] < base_strict['max_dd'] and final_metric['relative_return'] >= 30.0:
    decision = 'Diagnostic keep / no runtime promotion: conditional volatility targeting improves strict-OOS MaxDD but still misses the 15% target or walk-forward guard.'
else:
    decision = 'Diagnostic drop / no runtime promotion: conditional volatility targeting does not improve the remaining drawdown frontier enough.'

selection_payload = {
    'card': CARD,
    'description': 'Extreme-only conditional volatility target on top of recovery_reentry_regime_drawdown_guard_v1.',
    'baseline_card': BASELINE_CARD,
    'selected_candidate_id': int(selected['candidate_id']),
    'selected_params': selected_params,
    'baseline_strict': base_strict,
    'baseline_drawdown_event': base_event,
    'selected_strict': final_metric,
    'selected_drawdown_event': final_event,
    'strict_audit': final_audit,
    'strict_pass': strict_pass,
    'wf_guard_ok': wf_ok,
    'selected_wf': selected_wf,
    'comparisons': {
        'regime_drawdown_guard': load_json(BASELINE_JSON).get('selected_strict'),
        'cppi_floor': load_json(CPPI_JSON).get('selected_strict'),
        'trendbreak_cap': load_json(TRENDBREAK_JSON).get('selected_strict'),
    },
    'decision': decision,
    'artifacts': {
        'notebook': str((CARD_DIR / f'{CARD}.ipynb').relative_to(ROOT)),
        'parameter_screen': str(parameter_screen_path.relative_to(ROOT)),
        'parameter_screen_top': str(top_path.relative_to(ROOT)),
        'selection_table': str(selection_path.relative_to(ROOT)),
        'selected_daily': str(final_daily_path.relative_to(ROOT)),
        'walkforward_screen': str(wf_screen_path.relative_to(ROOT)),
        'walkforward_summary': str(wf_summary_path.relative_to(ROOT)),
    },
}
json_path = CARD_DIR / f'{CARD}.json'
selection_json_path = CARD_DIR / 'selection.json'
json_path.write_text(json.dumps(selection_payload, indent=2, ensure_ascii=False) + '\n')
selection_json_path.write_text(json.dumps(selection_payload, indent=2, ensure_ascii=False) + '\n')

wf_lines = []
for row in selected_wf:
    wf_lines.append(
        f"| {row['wf']} | {row['base_return']:.2f}% / {row['base_max_dd']:.2f}% | {row['return']:.2f}% / {row['max_dd']:.2f}% | {row['delta_return_vs_base']:.2f}pp / {row['delta_max_dd_vs_base']:.2f}pp | {row['changed_days']} |"
    )

readout = [
    '## Final Readout',
    '',
    'This notebook is the primary single-run conclusion artifact.',
    '',
    f"- Selected candidate: `{int(selected['candidate_id'])}` with params `{json.dumps(selected_params, sort_keys=True)}`",
    f"- Strict OOS baseline: `{base_strict['return']:.2f}% / {base_strict['sharpe']:.3f} / {base_strict['max_dd']:.2f}% MaxDD / excess {base_strict['relative_return']:.2f}pp`",
    f"- Strict OOS conditional vol target: `{final_metric['return']:.2f}% / {final_metric['sharpe']:.3f} / {final_metric['max_dd']:.2f}% MaxDD / excess {final_metric['relative_return']:.2f}pp`",
    f"- MaxDD event baseline: `{base_event['peak']} -> {base_event['trough']} ({base_event['max_dd']:.2f}%)`",
    f"- MaxDD event selected: `{final_event['peak']} -> {final_event['trough']} ({final_event['max_dd']:.2f}%)`",
    f"- Screen rows: `{len(screen)}`; strict target-pass rows: `{int(screen['passes_15dd_gate'].sum())}`; relative-pass rows: `{int(screen['passes_relative_gate'].sum())}`",
    f"- Decision: {decision}",
    '',
    '| window | baseline return / MaxDD | selected return / MaxDD | delta return / delta MaxDD | changed days |',
    '| --- | ---: | ---: | ---: | ---: |',
    *wf_lines,
    '',
    '### Interpretation',
    '',
    '- Conditional volatility targeting is useful only if the remaining drawdown is preceded by observable volatility extremes. If the selected candidate misses the 15% gate, this blocks another volatility-threshold tuning pass.',
    '- If the selected candidate mostly shifts the MaxDD event without reducing it materially, the next valid mechanism is not another cash-only cap; it likely needs an explicit hedge/alternative asset, or a training objective that learns lower peak exposure before volatility is realized.',
]
display(Markdown('\n'.join(readout)))
display(Markdown('## Top Strict Candidates'))
display(top[['candidate_id','vol_kind','lookback','target_vol','high_vol_pct','min_vix_pct','min_position','min_price_drawdown_21','min_scale','hold_days','return','relative_return','max_dd','changed_days','strict_score','maxdd_peak','maxdd_trough']])
display(Markdown('## Selection Table'))
display(selection[['candidate_id','return','relative_return','max_dd','passes_15dd_gate','wf_guard_ok','changed_days','wf_changed_days','wf_return_penalty','wf_dd_penalty','final_score','maxdd_peak','maxdd_trough']])
display(Markdown('## Walk-Forward Selected Summary'))
display(selected_wf_df)
print(json.dumps(selection_payload, indent=2, ensure_ascii=False))

baseline strict {'label': 'recovery_reentry_regime_drawdown_guard_v1', 'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 2272.8530194145196, 'buy_and_hold_return': 816.5124997770645, 'relative_return': 1456.3405196374551, 'sharpe': 1.2480221658238784, 'max_dd': 32.120179808738435, 'avg_pos': 0.4595798148626328, 'trades': 212}
baseline maxdd event {'peak': '2021-04-05', 'trough': '2021-05-12', 'max_dd': 32.120179808738435, 'peak_position': 0.7186343814, 'trough_position': 0.8380035218}


parameter_screen rows=8748 target15=0 relative_pass=7558 improved_dd=120


,candidate_id,vol_kind,lookback,target_vol,high_vol_pct,min_vix_pct,min_position,min_price_drawdown_21,min_scale,hold_days,return,relative_return,max_dd,changed_days,strict_score,maxdd_peak,maxdd_trough
0,1978,realized,21,0.35,0.75,NaN,0.55,0.0,0.40,5,2217.722707,1401.210207,30.923275,149,229.712809,2024-09-26,2025-04-08
1,2464,realized,21,0.45,0.75,NaN,0.55,0.0,0.40,5,2215.647179,1399.134679,30.923275,149,229.743942,2024-09-26,2025-04-08
2,1492,realized,21,0.25,0.75,NaN,0.55,0.0,0.40,5,2212.752147,1396.239647,30.923275,149,229.787368,2024-09-26,2025-04-08
3,1975,realized,21,0.35,0.75,NaN,0.55,0.0,0.25,5,2192.446780,1375.934280,30.923275,149,230.091948,2024-09-26,2025-04-08
4,1489,realized,21,0.25,0.75,NaN,0.55,0.0,0.25,5,2186.946930,1370.434430,30.923275,149,230.174446,2024-09-26,2025-04-08
5,1972,realized,21,0.35,0.75,NaN,0.55,0.0,0.00,5,2183.484259,1366.971759,30.923275,149,230.226386,2024-09-26,2025-04-08
6,1973,realized,21,0.35,0.75,NaN,0.55,0.0,0.00,10,3026.708098,2210.195598,30.923275,184,230.285855,2024-09-26,2025-04-08
7,1979,realized,21,0.35,0.75,NaN,0.55,0.0,0.40,10,2988.234589,2171.722089,30.923275,184,230.285855,2024-09-26,2025-04-08
8,1487,realized,21,0.25,0.75,NaN,0.55,0.0,0.00,10,3168.096090,2351.583590,30.923275,184,230.285855,2024-09-26,2025-04-08
9,1490,realized,21,0.25,0.75,NaN,0.55,0.0,0.25,10,3184.211255,2367.698755,30.923275,184,230.285855,2024-09-26,2025-04-08


selected 1979 {'vol_kind': 'realized', 'lookback': 21, 'target_vol': 0.35, 'high_vol_pct': 0.75, 'min_vix_pct': None, 'min_position': 0.55, 'min_price_drawdown_21': 0.0, 'min_scale': 0.4, 'trade_band': 0.02, 'hold_days': 10}


,candidate_id,return,relative_return,max_dd,passes_15dd_gate,wf_guard_ok,changed_days,wf_changed_days,wf_return_penalty,wf_dd_penalty,final_score,maxdd_peak,maxdd_trough
0,1979,2988.234589,2171.722089,30.923275,False,False,184,83,4.483905,0.000000,379.948440,2024-09-26,2025-04-08
1,2465,2897.179460,2080.666960,30.923275,False,False,184,83,4.483905,0.000000,379.948440,2024-09-26,2025-04-08
2,1493,3029.947185,2213.434686,30.923275,False,False,184,83,4.483905,0.000000,379.948440,2024-09-26,2025-04-08
3,1490,3184.211255,2367.698755,30.923275,False,False,184,83,5.756228,0.648002,382.731297,2024-09-26,2025-04-08
4,1976,3036.915172,2220.402672,30.923275,False,False,184,83,5.756228,0.648002,382.731297,2024-09-26,2025-04-08
5,1973,3026.708098,2210.195598,30.923275,False,False,184,83,6.197565,1.077985,384.517430,2024-09-26,2025-04-08
6,1487,3168.096090,2351.583590,30.923275,False,False,184,83,6.687199,1.555024,386.499030,2024-09-26,2025-04-08
7,2459,2861.786550,2045.274050,31.252842,False,False,184,83,5.894738,0.782949,387.905800,2021-02-16,2021-05-12
8,2462,2861.786550,2045.274050,31.252842,False,False,184,83,5.894738,0.782949,387.905800,2021-02-16,2021-05-12
9,1978,2217.722707,1401.210207,30.923275,False,False,149,63,7.373481,2.223651,388.853434,2024-09-26,2025-04-08


...

## Selection Table

,candidate_id,return,relative_return,max_dd,passes_15dd_gate,wf_guard_ok,changed_days,wf_changed_days,wf_return_penalty,wf_dd_penalty,final_score,maxdd_peak,maxdd_trough
0,1979,2988.234589,2171.722089,30.923275,False,False,184,83,4.483905,0.000000,379.948440,2024-09-26,2025-04-08
1,2465,2897.179460,2080.666960,30.923275,False,False,184,83,4.483905,0.000000,379.948440,2024-09-26,2025-04-08
2,1493,3029.947185,2213.434686,30.923275,False,False,184,83,4.483905,0.000000,379.948440,2024-09-26,2025-04-08
3,1490,3184.211255,2367.698755,30.923275,False,False,184,83,5.756228,0.648002,382.731297,2024-09-26,2025-04-08
4,1976,3036.915172,2220.402672,30.923275,False,False,184,83,5.756228,0.648002,382.731297,2024-09-26,2025-04-08
5,1973,3026.708098,2210.195598,30.923275,False,False,184,83,6.197565,1.077985,384.517430,2024-09-26,2025-04-08
6,1487,3168.096090,2351.583590,30.923275,False,False,184,83,6.687199,1.555024,386.499030,2024-09-26,2025-04-08
7,2459,2861.786550,2045.274050,31.252842,False,False,184,83,5.894738,0.782949,387.905800,2021-02-16,2021-05-12
8,2462,2861.786550,2045.274050,31.252842,False,False,184,83,5.894738,0.782949,387.905800,2021-02-16,2021-05-12
9,1978,2217.722707,1401.210207,30.923275,False,False,149,63,7.373481,2.223651,388.853434,2024-09-26,2025-04-08


## Walk-Forward Selected Summary

,wf,label,rows,start,end,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,trades,trigger_days,active_days,changed_days,avg_scale_when_active,base_return,base_max_dd,delta_return_vs_base,delta_max_dd_vs_base,json_path,daily_path
0,wf_2020,condvol_realized21_tv0.35_pct0.75_dd0.00_scale...,252,2020-01-02,2020-12-30,113.889962,58.093571,55.796392,1.582831,26.139320,0.423880,29,0,0,0,1.000000,113.889962,26.139320,0.000000,-1.065814e-14,runtime/execution/card/recovery_reentry_condit...,runtime/execution/card/recovery_reentry_condit...
1,wf_2022,condvol_realized21_tv0.35_pct0.75_dd0.00_scale...,250,2022-01-03,2022-12-29,-21.610018,-86.588072,64.978054,-0.784975,23.626625,0.165095,16,8,17,28,0.400000,-17.126113,24.218215,-4.483905,-5.915899e-01,runtime/execution/card/recovery_reentry_condit...,runtime/execution/card/recovery_reentry_condit...
2,wf_2023_2024,condvol_realized21_tv0.35_pct0.75_dd0.00_scale...,251,2024-01-02,2024-12-30,68.533647,-0.356635,68.890281,1.698616,16.826953,0.259105,38,26,55,55,0.417938,58.194746,30.183184,10.338901,-1.335623e+01,runtime/execution/card/recovery_reentry_condit...,runtime/execution/card/recovery_reentry_condit...


{
  "card": "recovery_reentry_conditional_vol_target_v1",
  "description": "Extreme-only conditional volatility target on top of recovery_reentry_regime_drawdown_guard_v1.",
  "baseline_card": "recovery_reentry_regime_drawdown_guard_v1",
  "selected_candidate_id": 1979,
  "selected_params": {
    "vol_kind": "realized",
    "lookback": 21,
    "target_vol": 0.35,
    "high_vol_pct": 0.75,
    "min_vix_pct": null,
    "min_position": 0.55,
    "min_price_drawdown_21": 0.0,
    "min_scale": 0.4,
    "trade_band": 0.02,
    "hold_days": 10
  },
  "baseline_strict": {
    "label": "recovery_reentry_regime_drawdown_guard_v1",
    "rows": 1605,
    "start": "2020-01-02",
    "end": "2026-05-21",
    "return": 2272.8530194145196,
    "buy_and_hold_return": 816.5124997770645,
    "relative_return": 1456.3405196374551,
    "sharpe": 1.2480221658238784,
    "max_dd": 32.120179808738435,
    "avg_pos": 0.4595798148626328,
    "trades": 212
  },
  "baseline_drawdown_event": {
    "peak": "2021-04-